
## Naive Bayes from Scratch: Delivery Delay Prediction



## Dataset

Each row describes one delivery route. Unlike a binary-feature dataset, these predictors contain text categories with two or three possible values.

| Attribute | Possible values |
|---|---|
| `weather` | `clear`, `fog`, `rain` |
| `traffic` | `low`, `medium`, `high` |
| `distance_band` | `short`, `medium`, `long` |
| `vehicle_type` | `bike`, `van`, `truck` |
| `service_level` | `regular`, `express` |
| `day_type` | `weekday`, `weekend` |

The target is stored in `delivery_status`, and `partition` provides the fixed split.

In [1]:
%pip install numpy pandas


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

delivery_columns = [
    "weather",
    "traffic",
    "distance_band",
    "vehicle_type",
    "service_level",
    "day_type",
    "delivery_status",
    "partition",
]

delivery_rows = [
    ["rain", "high", "long", "van", "regular", "weekday", "late", "train"],
    ["rain", "high", "medium", "bike", "express", "weekday", "late", "train"],
    ["fog", "medium", "long", "truck", "regular", "weekday", "late", "train"],
    ["rain", "medium", "long", "truck", "express", "weekend", "late", "train"],
    ["clear", "high", "long", "van", "regular", "weekday", "late", "train"],
    ["fog", "high", "medium", "bike", "regular", "weekend", "late", "train"],
    ["rain", "low", "long", "truck", "regular", "weekend", "late", "train"],
    ["clear", "medium", "long", "bike", "express", "weekday", "late", "train"],
    ["fog", "high", "long", "van", "express", "weekday", "late", "train"],
    ["rain", "medium", "medium", "van", "regular", "weekend", "late", "train"],
    ["clear", "low", "short", "bike", "regular", "weekday", "on_time", "train"],
    ["clear", "low", "medium", "van", "express", "weekday", "on_time", "train"],
    ["clear", "medium", "short", "van", "regular", "weekday", "on_time", "train"],
    ["fog", "low", "short", "bike", "express", "weekend", "on_time", "train"],
    ["clear", "low", "long", "truck", "regular", "weekend", "on_time", "train"],
    ["rain", "low", "short", "van", "express", "weekend", "on_time", "train"],
    ["clear", "medium", "medium", "truck", "express", "weekday", "on_time", "train"],
    ["fog", "medium", "short", "van", "regular", "weekday", "on_time", "train"],
    ["clear", "low", "short", "truck", "express", "weekend", "on_time", "train"],
    ["rain", "medium", "short", "bike", "regular", "weekday", "on_time", "train"],
    ["fog", "low", "medium", "van", "express", "weekday", "on_time", "train"],
    ["clear", "medium", "short", "bike", "express", "weekend", "on_time", "train"],
    ["rain", "low", "medium", "truck", "regular", "weekday", "on_time", "train"],
    ["fog", "medium", "medium", "truck", "regular", "weekend", "on_time", "train"],
    ["rain", "high", "long", "truck", "regular", "weekday", "late", "test"],
    ["fog", "high", "medium", "van", "express", "weekend", "late", "test"],
    ["clear", "high", "long", "bike", "regular", "weekday", "late", "test"],
    ["rain", "medium", "long", "van", "express", "weekend", "late", "test"],
    ["clear", "low", "short", "van", "regular", "weekday", "on_time", "test"],
    ["fog", "low", "medium", "bike", "express", "weekday", "on_time", "test"],
    ["rain", "low", "short", "truck", "express", "weekend", "on_time", "test"],
    ["clear", "medium", "medium", "van", "regular", "weekend", "on_time", "test"],
]

deliveries = pd.DataFrame(delivery_rows, columns=delivery_columns)
deliveries

,weather,traffic,distance_band,vehicle_type,service_level,day_type,delivery_status,partition
0,rain,high,long,van,regular,weekday,late,train
1,rain,high,medium,bike,express,weekday,late,train
2,fog,medium,long,truck,regular,weekday,late,train
3,rain,medium,long,truck,express,weekend,late,train
4,clear,high,long,van,regular,weekday,late,train
5,fog,high,medium,bike,regular,weekend,late,train
6,rain,low,long,truck,regular,weekend,late,train
7,clear,medium,long,bike,express,weekday,late,train
8,fog,high,long,van,express,weekday,late,train
9,rain,medium,medium,van,regular,weekend,late,train


In [3]:
route_attributes = [
    "weather",
    "traffic",
    "distance_band",
    "vehicle_type",
    "service_level",
    "day_type",
]

learning_routes = deliveries[deliveries["partition"] == "train"].reset_index(drop=True)
evaluation_routes = deliveries[deliveries["partition"] == "test"].reset_index(drop=True)

route_train = learning_routes[route_attributes]
status_train = learning_routes["delivery_status"]
route_test = evaluation_routes[route_attributes]
status_test = evaluation_routes["delivery_status"]
status_labels = np.sort(status_train.unique())

print("Learning routes:", route_train.shape)
print("Evaluation routes:", route_test.shape)
print(status_train.value_counts())

Learning routes: (24, 6)
Evaluation routes: (8, 6)
delivery_status
on_time    14
late       10
Name: count, dtype: int64


## Base rates

Complete `compute_base_rates`. It must return a dictionary whose keys are the class labels and whose values are $P(C=c)$. Do not use `value_counts(normalize=True)`.

$$P(C=c)=\frac{\text{number of training routes in class }c}{\text{number of training routes}}$$

In [4]:
def compute_base_rates(outcomes):
    """Calculate one prior probability for each observed class.

    Parameters
    ----------
    outcomes : pandas.Series
        Training class labels.

    Returns
    -------
    dict
        Mapping from a class label to its prior probability.
    """
    rates = {}

    for outcome in np.sort(outcomes.unique()):
        outcome_count = (outcomes == outcome).sum()
        rates[outcome] = outcome_count / len(outcomes)

    return rates


base_rates = compute_base_rates(status_train)
base_rates

{'late': np.float64(0.4166666666666667),
 'on_time': np.float64(0.5833333333333334)}

## Long-form conditional table

For every class, attribute, and observed category, calculate the Laplace-smoothed probability:

$$P(X_j=v\mid C=c)=\frac{N_{j,v,c}+\alpha}{N_c+\alpha K_j}$$

Complete `build_conditional_table`. It must return a long-form DataFrame with the columns `class_label`, `attribute`, `category`, and `probability`. Here, $\alpha=1$ and $K_j$ is the number of categories of attribute $j$ in the full training set.

In [5]:
def build_conditional_table(inputs, outcomes, labels, alpha=1.0):
    """Build a long-form table of smoothed conditional probabilities.

    Parameters
    ----------
    inputs : pandas.DataFrame
        Discrete training attributes.
    outcomes : pandas.Series
        Training labels aligned with inputs.
    labels : array-like
        Class labels to process.
    alpha : float
        Laplace smoothing parameter.

    Returns
    -------
    pandas.DataFrame
        One row per class, attribute, and category.
    """
    category_map = {
        attribute: np.sort(inputs[attribute].unique())
        for attribute in inputs.columns
    }
    probability_rows = []

    for label in labels:
        class_inputs = inputs[outcomes == label]

        for attribute, categories in category_map.items():
            for category in categories:
                matching_count = (class_inputs[attribute] == category).sum()
                conditional_probability = (matching_count + alpha) / (len(class_inputs) 
                                                                      + alpha * len(categories))

                probability_rows.append(
                    {
                        "class_label": label,
                        "attribute": attribute,
                        "category": category,
                        "probability": conditional_probability,
                    }
                )

    return pd.DataFrame(probability_rows)


conditional_table = build_conditional_table(
    route_train, status_train, status_labels, alpha=1.0
)
conditional_table

,class_label,attribute,category,probability
0,late,weather,clear,0.230769
1,late,weather,fog,0.307692
2,late,weather,rain,0.461538
3,late,traffic,high,0.461538
4,late,traffic,low,0.153846
5,late,traffic,medium,0.384615
6,late,distance_band,long,0.615385
7,late,distance_band,medium,0.307692
8,late,distance_band,short,0.076923
9,late,vehicle_type,bike,0.307692


## Route scoring and decisions

Complete `score_routes`. For each route and class, start with the log base rate, look up every required probability in `conditional_table`, and add its logarithm.

$$L(c)=\log P(C=c)+\sum_j\log P(X_j=x_j\mid C=c)$$

Then create `route_decisions` as a Python list containing the class with the highest score for each evaluation route.

In [6]:
def score_routes(records, labels, rates, probability_table):
    """Calculate class log-scores for a collection of delivery routes.

    Parameters
    ----------
    records : pandas.DataFrame
        Delivery routes to score.
    labels : array-like
        Candidate class labels.
    rates : dict
        Prior probabilities keyed by class.
    probability_table : pandas.DataFrame
        Long-form conditional-probability table.

    Returns
    -------
    pandas.DataFrame
        Rows represent routes and columns represent class log-scores.
    """
    score_sheet = pd.DataFrame(index=records.index, columns=labels, dtype=float)

    for record_index, record in records.iterrows():
        for label in labels:
            route_score = np.log(rates[label])

            for attribute, observed_category in record.items():
                matching_row = probability_table[
                    (probability_table["class_label"] == label)
                    & (probability_table["attribute"] == attribute)
                    & (probability_table["category"] == observed_category)
                ]

                probability_used = matching_row["probability"].values[0]
                route_score = route_score + np.log(probability_used)

            score_sheet.loc[record_index, label] = route_score

    return score_sheet


route_log_scores = score_routes(
    route_test, status_labels, base_rates, conditional_table
)

route_decisions = route_log_scores.idxmax(axis=1)

route_report = route_test.copy()
route_report["actual_status"] = status_test.to_numpy()
route_report["predicted_status"] = route_decisions
route_report

,weather,traffic,distance_band,vehicle_type,service_level,day_type,actual_status,predicted_status
0,rain,high,long,truck,regular,weekday,late,late
1,fog,high,medium,van,express,weekend,late,late
2,clear,high,long,bike,regular,weekday,late,late
3,rain,medium,long,van,express,weekend,late,late
4,clear,low,short,van,regular,weekday,on_time,on_time
5,fog,low,medium,bike,express,weekday,on_time,on_time
6,rain,low,short,truck,express,weekend,on_time,on_time
7,clear,medium,medium,van,regular,weekend,on_time,on_time


## Delivery result grid 

Complete `build_result_grid` without using `pd.crosstab` or a metrics library. Rows must be **predicted** classes and columns must be **actual** classes. Also calculate `success_rate`, the fraction of correct route decisions.

In [7]:
def build_result_grid(actual_values, predicted_values, label_order):
    """Count delivery outcomes using predicted rows and actual columns.

    Parameters
    ----------
    actual_values : array-like
        Actual delivery statuses.
    predicted_values : array-like
        Predicted delivery statuses.
    label_order : array-like
        Row and column order for the result grid.

    Returns
    -------
    pandas.DataFrame
        Count grid with predicted rows and actual columns.
    """
    grid = np.zeros((len(label_order), len(label_order)), dtype=int)
    label_position = {label: i for i, label in enumerate(label_order)}

    for actual, predicted in zip(actual_values, predicted_values):
        # TODO
        pass

    return pd.DataFrame(
        grid,
        index=[f"predicted_{label}" for label in label_order],
        columns=[f"actual_{label}" for label in label_order],
    )


success_rate = (status_test == route_decisions).mean()
delivery_grid = build_result_grid(
    status_test, route_decisions, status_labels
)

print(f"Success rate: {success_rate:.3f}")
display(delivery_grid)

Success rate: 1.000


,actual_late,actual_on_time
predicted_late,0,0
predicted_on_time,0,0
